# 2장 실습. Jev API: 상태·질문 스키마와 응답 유형

Jev 에 판단을 맡기는 요청은 세 필드뿐이다. 판단할 대상 `state`, 모델 이름 `model`, 묻고 싶은 것을 이름 붙여 모은 `questions` 다. 이 노트북은 그 요청을 만들고, 돌아온 확률을 읽고, 이 책 전체가 쓰는 `ask()` 를 손에 익힌다.

| 절 | 내용 |
|---|---|
| 2.1 | choice 호출 |
| 2.2 | 확률 분포와 confidence 필드 |
| 2.3 | noul 응답 |
| 2.4 | 질문 문장 민감도 |
| 2.5 | 사전 검증과 오류 응답 |
| 2.6 | 질문 배칭 |
| 2.7 | 버전 고정과 결정 로그 |
| 2.8 | 비용 추정 |

`jupyter-new/.env` 에 `TYPESAFE_API_KEY` 가 있어야 한다(0장). Jev 에 요청 20건 남짓을 보낸다.

In [1]:
import os, json, hashlib, datetime
import requests
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
http = requests.Session()
retry = HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[408, 429, 500, 502, 503, 504]))
http.mount("https://", retry); http.mount("http://", retry)

def post(state, questions, model="jev-latest"):
    return http.post(f"{JEV_URL}/v1/systemone", timeout=60,
                     headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
                     json={"model": model, "state": state, "questions": questions})

def ask(state, questions, model="jev-latest"):
    r = post(state, questions, model)
    r.raise_for_status()
    return r.json() | {"_latency_s": r.elapsed.total_seconds()}

`post()` 는 HTTP 응답을 그대로 돌려주고, `ask()` 는 오류면 예외를 내고 아니면 본문에 응답 시간 `_latency_s` 를 붙여 돌려준다. 재시도는 세션에 붙인 `Retry` 가 맡는다. 408·429·5xx 는 0.5초에서 시작해 두 배씩 기다리며 네 번까지 다시 보낸다. 400·401·422 는 요청이 잘못된 것이라 다시 보내지 않는다.

질문 셋도 짧은 함수로 만든다.

In [2]:
def choice(instructions, criteria):
    return {"type": "choice", "instructions": instructions, "criteria": criteria}

def score(instructions, levels):
    return {"type": "score", "instructions": instructions, "criteria": levels}

def noul(instructions, true="", false=""):
    return {"type": "noul", "instructions": instructions} | ({"criteria": {"true": true, "false": false}} if true else {})

## 2.1 choice 호출

고객 메시지 하나를 넣고 무엇을 원하는지 네 갈래 중에서 고르게 한다. `criteria` 에 선택지 이름과 설명을 함께 적는다. 이름은 코드가 분기에 쓰고, 설명은 모델이 읽는다.

In [3]:
state = "지난달 청구서 사본을 보내 주실 수 있나요? 회계팀에 제출해야 해요."
questions = {"request": choice("고객이 원하는 것", {
    "invoice_copy": "청구서·영수증 같은 문서의 사본을 원한다",
    "refund": "돈을 돌려받고 싶다",
    "cancel": "구독을 끝내고 싶다",
    "other": "위 어느 것도 아니다"})}
res = ask(state, questions)
print(json.dumps(res, ensure_ascii=False, indent=2))

{
  "model": "jev-1.13.0",
  "answers": {
    "request": {
      "type": "choice",
      "choice": "invoice_copy",
      "confidence": 1.0,
      "probabilities": {
        "invoice_copy": 1.0,
        "refund": 0.0,
        "cancel": 0.0,
        "other": 0.0
      }
    }
  },
  "usage": {
    "input_tokens": 391,
    "output_tokens": 47
  },
  "_latency_s": 0.212623
}


응답에 문장은 없다. `answers` 아래에 질문 이름과 같은 키가 있고, 그 안에 고른 답(`choice`), 선택지별 확률(`probabilities`), 확신(`confidence`)이 들어 있다. `usage` 의 출력 토큰은 과금되지 않는다.

요청은 `jev-latest` 로 보냈다. 응답의 `model` 에는 이 별칭이 실제로 가리킨 버전이 적혀 있다.

## 2.2 확률 분포와 confidence 필드

이 책의 러닝 예제다. 결제가 두 번 된 고객이 오늘 안에 처리되지 않으면 해지하겠다고 한다. 질문 셋을 한 요청에 묶어 묻는다.

`confidence` 는 분포가 한곳에 얼마나 몰렸는지를 숫자 하나로 줄인 값이다. Jev 문서가 3지선다에 대해 밝힌 `(3 × 최대 확률 − 1) / 2` 를 K 지선다로 넓히면 `(K × 최대 확률 − 1) / (K − 1)` 이다. Laya 는 같은 이름에 `1 − 엔트로피 / log K` 를 쓴다(10장). 응답 값은 반올림돼 있어 0.01 안팎 어긋날 수 있다.

In [4]:
import numpy as np

TICKET_STATE = {"channel": "앱 문의",
                "message": "결제가 두 번 됐어요. 어제 저녁에 한 번만 결제했는데 카드 문자가 두 번 왔습니다. 오늘 안에 처리 안 되면 해지할게요."}
TICKET_QUESTIONS = {
    "department": choice("이 문의를 처리할 팀", {"billing": "결제, 환불, 청구 오류", "technical": "앱 오류, 로그인, 연동 문제",
                                             "account": "가입, 해지, 요금제 변경", "general": "그 밖의 문의"}),
    "urgency": score("얼마나 급하게 처리해야 하는가", ["일반 처리로 충분하다", "오늘 안에 처리해야 한다", "즉시 사람이 연락해야 한다"]),
    "churn_risk": noul("고객이 해지를 언급하거나 해지할 뜻을 보인다",
                       true="해지·탈퇴·이용 중단을 직접 말하거나 조건부로 예고한다", false="해지에 관한 언급이 없다"),
}
res_ko = ask(TICKET_STATE, TICKET_QUESTIONS)
print(json.dumps(res_ko["answers"], ensure_ascii=False, indent=1))

jev_conf = lambda p: (len(p) * max(p) - 1) / (len(p) - 1)
entropy_conf = lambda p: 1 + sum(x * np.log(x) for x in p if x > 0) / np.log(len(p))
for name in ["department", "urgency"]:
    a = res_ko["answers"][name]
    p = list(a["probabilities"].values())
    print(f"[{name}] 보고값 {a['confidence']} | (K·max−1)/(K−1) = {jev_conf(p):.4f} | 1−H/logK = {entropy_conf(p):.4f}")

urg = res_ko["answers"]["urgency"]
print(f"[urgency] score {urg['score']} | 확률 가중 평균 {sum(int(k) * v for k, v in urg['probabilities'].items()):.4f}")

{
 "department": {
  "type": "choice",
  "choice": "billing",
  "confidence": 1.0,
  "probabilities": {
   "billing": 1.0,
   "technical": 0.0,
   "account": 0.0,
   "general": 0.0
  }
 },
 "urgency": {
  "type": "score",
  "score": 1.06,
  "confidence": 0.91,
  "legend": {
   "0": "일반 처리로 충분하다",
   "1": "오늘 안에 처리해야 한다",
   "2": "즉시 사람이 연락해야 한다"
  },
  "probabilities": {
   "0": 0.0,
   "1": 0.94,
   "2": 0.06
  }
 },
 "churn_risk": {
  "type": "noul",
  "noul": 0.99
 }
}
[department] 보고값 1.0 | (K·max−1)/(K−1) = 1.0000 | 1−H/logK = 1.0000
[urgency] 보고값 0.91 | (K·max−1)/(K−1) = 0.9100 | 1−H/logK = 0.7934
[urgency] score 1.06 | 확률 가중 평균 1.0600


score 의 `score` 는 단계 번호를 확률로 가중 평균한 값이다. 1.5 처럼 두 단계 사이에 놓일 수 있으므로, 어느 단계로 볼지는 코드에서 정한다.

score 의 `confidence` 는 공개된 식이 없다. 그래서 이 책은 엔진을 비교하거나 게이트를 걸 때 `confidence` 필드 대신 **고른 답의 확률(최대 확률)** 을 쓴다. 8장의 평가 하네스도 그렇게 잰다.

## 2.3 noul 응답

noul 은 주장 하나가 참일 확률만 돌려준다. 확률 자체가 확신이므로 게이트에는 0.5 에서 얼마나 떨어졌는지, 곧 `max(p, 1−p)` 를 쓴다.

In [5]:
a = ask("검토 끝났습니다. 금요일까지 계약서 보내 주시면 바로 진행하겠습니다.",
        {"ready_to_buy": noul("보낸 사람이 구매를 결정했다", true="진행하겠다고 말한다", false="아직 검토 중이거나 거절한다")})
p = a["answers"]["ready_to_buy"]["noul"]
print(a["answers"], "| 판단:", p >= 0.5, "| 확신 max(p, 1−p):", max(p, 1 - p))

{'ready_to_buy': {'type': 'noul', 'noul': 0.7}} | 판단: True | 확신 max(p, 1−p): 0.7


## 2.4 질문 문장 민감도

같은 메시지에 같은 뜻의 질문을 두 가지로 묻는다. 하나는 이중 부정에 기준이 없고, 다른 하나는 주장 하나에 참·거짓 기준을 붙였다. 튜토리얼(AY Automate)의 영어 예시와 그 한국어 번역을 함께 보낸다. 튜토리얼 저자는 2026년 9월 20일 영어 예시에서 0.54 와 0.41 을 받았다고 적었다.

In [6]:
cases = {
    "영어": ({"message": "I would rather not cancel, but I can't afford this anymore."},
             noul("Is it not the case that the customer does not want to keep the plan?"),
             noul("Is the customer asking to cancel?", true="The customer says they are cancelling or wants to cancel",
                  false="The customer wants to keep the plan or is only asking a question")),
    "한국어": ({"message": "해지하고 싶지는 않은데, 이제 요금을 감당할 수가 없어요."},
              noul("고객이 요금제를 유지하고 싶어 하지 않는다는 것은 사실이 아니다"),
              noul("고객이 해지를 요청한다", true="고객이 해지하겠다고 말하거나 해지를 원한다고 밝힌다",
                   false="고객이 요금제를 유지하고 싶어 하거나 질문만 한다")),
}
for lang, (msg, neg, plain) in cases.items():
    r1, r2 = ask(msg, {"cancel": neg}), ask(msg, {"cancel": plain})
    print(f"[{lang}] 이중 부정, 기준 없음 : noul={r1['answers']['cancel']['noul']:.2f}  input_tokens={r1['usage']['input_tokens']}")
    print(f"[{lang}] 주장 하나 + 기준     : noul={r2['answers']['cancel']['noul']:.2f}  input_tokens={r2['usage']['input_tokens']}")

[영어] 이중 부정, 기준 없음 : noul=0.55  input_tokens=302
[영어] 주장 하나 + 기준     : noul=0.47  input_tokens=332
[한국어] 이중 부정, 기준 없음 : noul=0.83  input_tokens=319
[한국어] 주장 하나 + 기준     : noul=0.08  input_tokens=365


메시지는 「해지하고 싶지 않지만 더는 감당이 안 된다」로 실제로 엇갈린다. 어느 확률도 정답이라 할 수 없다. 다만 두 번째 질문은 무엇을 물었는지 동료에게 설명할 수 있다. 같은 요청을 여러 번 보내면 얼마나 흔들리는지는 4장에서 잰다.

## 2.5 사전 검증과 오류 응답

API 가 거절하는 요청이 있고, 받아 주지만 쓸모없는 요청도 있다. 보내기 전에 거르는 검사를 표 하나로 만든다. 검사 이름과 「걸리는 조건」을 짝지어 두면 조건을 늘릴 때 줄 하나만 더한다.

In [7]:
def validate(questions):
    rules = {
        "questions 가 비어 있다": lambda qs: not qs,
        "알 수 없는 type": lambda qs: any(q["type"] not in ("choice", "score", "noul") for q in qs.values()),
        "choice 선택지가 둘 미만": lambda qs: any(q["type"] == "choice" and len(q["criteria"]) < 2 for q in qs.values()),
        "score 단계가 2~10 밖": lambda qs: any(q["type"] == "score" and not 2 <= len(q["criteria"]) <= 10 for q in qs.values()),
    }
    return [name for name, broken in rules.items() if broken(questions)]

bad = {"빈 questions": {},
       "모르는 type": {"q": {"type": "essay", "instructions": "이 글에 대해 써라."}},
       "단계 하나짜리 score": {"q": score("이 글은 얼마나 친절한가", ["단계 하나뿐"])}}
for name, qs in bad.items():
    r = post("안녕하세요", qs)
    print(f"[{name}] 로컬 검사: {validate(qs) or '통과'} | API: HTTP {r.status_code} {r.text[:90]}")

[빈 questions] 로컬 검사: ['questions 가 비어 있다'] | API: HTTP 422 {"detail":[{"type":"too_short","loc":["body","questions"],"msg":"Dictionary should have at
[모르는 type] 로컬 검사: ['알 수 없는 type'] | API: HTTP 400 {"detail":{"error_type":"api_usage_error","message":"Invalid request."}}
[단계 하나짜리 score] 로컬 검사: ['score 단계가 2~10 밖'] | API: HTTP 200 {"model":"jev-1.13.0","answers":{"q":{"type":"score","score":0.0,"confidence":1.0,"legend"


셋째가 가장 위험하다. API 는 오류 없이 답을 주지만, 단계가 하나뿐이니 아무 정보가 없는 답이 확신 1.0 으로 나올 수 있다. `validate()` 는 이런 질문을 보내기 전에 막는다. 여기서는 오류 응답을 보려고 `ask()` 대신 `post()` 로 보냈다. 실제 코드는 `ask()` 를 쓰고, 오류는 예외로 드러나게 둔다.

## 2.6 질문 배칭

2.2절은 질문 셋을 한 요청에 묶었다. 묶으면 `state` 를 한 번만 읽으므로 입력 토큰이 한 번만 과금된다. 같은 질문 셋을 따로 보내 비교한다.

In [8]:
import pandas as pd

apart = {n: ask(TICKET_STATE, {n: q}) for n, q in TICKET_QUESTIONS.items()}
brief = lambda a: a.get("choice") or f"{a.get('score', a.get('noul')):.2f}"
pd.DataFrame([{"방식": "한 요청에 셋", "입력 토큰": res_ko["usage"]["input_tokens"], "지연(초)": res_ko["_latency_s"],
               **{n: brief(res_ko["answers"][n]) for n in TICKET_QUESTIONS}},
              {"방식": "셋으로 나눔", "입력 토큰": sum(r["usage"]["input_tokens"] for r in apart.values()),
               "지연(초)": sum(r["_latency_s"] for r in apart.values()), **{n: brief(apart[n]["answers"][n]) for n in TICKET_QUESTIONS}}])

,방식,입력 토큰,지연(초),department,urgency,churn_risk
0,한 요청에 셋,565,0.171534,billing,1.06,0.99
1,셋으로 나눔,1211,0.517624,billing,1.05,0.99


## 2.7 버전 고정과 결정 로그

실험 중에는 `jev-latest` 를 쓰고, 운영에서는 버전 id 를 보낸다. 지난달에 맞춘 임계값이 새 버전에서도 맞는다는 보장이 없기 때문이다. 별칭이 지금 가리키는 버전을 응답에서 읽어 그대로 다시 보낸다.

In [9]:
pinned = res_ko["model"]
res_pin = ask(TICKET_STATE, TICKET_QUESTIONS, model=pinned)
print("jev-latest →", res_ko["model"], "| 고정 요청", pinned, "→", res_pin["model"])

jev-latest → jev-1.13.0 | 고정 요청 jev-1.13.0 → jev-1.13.0


판단마다 남길 기록이다. 8장의 평가 기록도 같은 필드를 쓴다.

In [10]:
def decision_log(state, res, action):
    return {"ts": datetime.datetime.now().isoformat(timespec="seconds"),
            "model": res["model"],                                    # 별칭이 아니라 실제 버전
            "state_sha": hashlib.sha256(json.dumps(state, ensure_ascii=False).encode()).hexdigest()[:12],
            "answers": res["answers"],                                # 쓰지 않은 답까지 전부 남긴다
            "input_tokens": res["usage"]["input_tokens"], "latency_s": round(res["_latency_s"], 3),
            "action": action, "outcome": None}                        # outcome 은 나중에 실제 결과로 채운다

print(json.dumps(decision_log(TICKET_STATE, res_pin, "billing_queue"), ensure_ascii=False, indent=1))

{
 "ts": "2026-10-16T21:22:35",
 "model": "jev-1.13.0",
 "state_sha": "529334a891da",
 "answers": {
  "department": {
   "type": "choice",
   "choice": "billing",
   "confidence": 1.0,
   "probabilities": {
    "billing": 1.0,
    "technical": 0.0,
    "account": 0.0,
    "general": 0.0
   }
  },
  "urgency": {
   "type": "score",
   "score": 1.05,
   "confidence": 0.93,
   "legend": {
    "0": "일반 처리로 충분하다",
    "1": "오늘 안에 처리해야 한다",
    "2": "즉시 사람이 연락해야 한다"
   },
   "probabilities": {
    "0": 0.0,
    "1": 0.95,
    "2": 0.05
   }
  },
  "churn_risk": {
   "type": "noul",
   "noul": 0.98
  }
 },
 "input_tokens": 565,
 "latency_s": 0.237,
 "action": "billing_queue",
 "outcome": null
}


## 2.8 비용 추정

Jev 는 입력 100만 토큰당 0.042달러이고 출력은 받지 않는다. 러닝 예제 한 건의 실제 입력 토큰으로 한 달 5만 건의 비용을 어림한다. 한국어는 같은 내용을 영어보다 많은 토큰으로 쪼개므로 한국어 요청으로 직접 잰다.

In [11]:
PRICE_PER_M = 0.042
tickets, tokens = 50_000, res_ko["usage"]["input_tokens"]
print(f"건당 입력 {tokens} 토큰 × 월 {tickets:,}건 = {tickets * tokens / 1e6:.1f}M 토큰 → ${tickets * tokens / 1e6 * PRICE_PER_M:.2f}")

건당 입력 565 토큰 × 월 50,000건 = 28.2M 토큰 → $1.19


## 정리

- 요청은 `state`·`model`·`questions` 세 필드다. 질문은 choice·score·noul 가운데 하나이고, 한 요청에 여러 개를 묶는다.
- choice 의 `confidence` 는 엔진마다 정의가 다르고 score 의 것은 공개된 식이 없다. 비교와 게이트에는 최대 확률을 쓴다.
- noul 은 확률 하나만 준다. 게이트에는 `max(p, 1−p)` 를 쓴다.
- API 가 받아 주는 요청이라도 쓸모없을 수 있다. 보내기 전에 `validate()` 로 거른다.
- 응답의 `model` 을 기록하고, 운영에서는 버전 id 를 고정한다.